# Creative Story Generator — Nova

**Lumexa AI Builder course**

A creative-writing assistant, **Nova**, that builds a structured story brief
from your inputs (genre, characters, setting, tone, length, theme) and
generates a complete original short story with a title.

This notebook is fully self-contained and works with **Runtime → Run all** —
no setup, no API keys required, and nothing to upload.

**What you'll do:**
1. Reuse Nova's original creative-writing persona and `build_story_prompt()`
   function verbatim.
2. Generate a full story from concrete example field values, displayed as
   Markdown.
3. Generate two versions at different "creativity" (temperature) settings
   and compare them, honoring the original temperature-slider idea.
4. Learn about an *optional* upgrade path: paste in a real OpenAI API key to
   let Nova write using `gpt-4o-mini` — completely optional.

**Why no required API key / no downloaded AI model?** So this notebook
always finishes `Runtime → Run all` for every student, instantly, with zero
setup friction. The optional OpenAI section shows what a real LLM writer
sounds like.


## Step 1: Nova's persona and the structured story-brief builder

Both `SYSTEM_PROMPT` and `build_story_prompt()` are reused verbatim from the
original `src/app.py` Streamlit app. They're used two ways here:

- As the literal system + user prompt if you opt in to the real OpenAI API.
- `build_story_prompt()`'s fields (genre, characters, setting, tone, theme)
  are also fed directly into our rule-based story generator below.


In [1]:
SYSTEM_PROMPT = """
You are Nova, a creative writing assistant aboard the Lumexa space station.
You write original, engaging short stories for young readers (ages 12-17)
based on a structured story brief provided by the user.

RULES:
- Write a complete, self-contained story with a clear beginning, middle,
  and end — never leave it obviously unfinished mid-scene.
- Start your response with a short, original title on its own line,
  formatted as "# Title", followed by the story itself.
- Match the requested genre, tone, and approximate length as closely as
  possible.
- Keep content appropriate for a general audience of young teens: avoid
  graphic violence, explicit content, or excessive gore, even for
  horror-genre requests — keep any scares mild and suspense-driven rather
  than graphic.
- Make creative, specific use of the provided characters, setting, and
  theme rather than generic filler description.
""".strip()

GENRES = [
    "Science Fiction", "Fantasy", "Mystery", "Adventure",
    "Comedy", "Horror (mild/age-appropriate)", "Slice of Life",
]
TONES = ["Lighthearted", "Serious", "Suspenseful", "Whimsical", "Inspirational"]
LENGTHS = {
    "Short (about 300 words)": 450,
    "Medium (about 600 words)": 900,
    "Long (about 1000 words)": 1500,
}


def build_story_prompt(genre, characters, setting, tone, length_label, theme):
    """Combine all form inputs into one structured, explicit story brief."""
    return f"""
Write an original short story with the following brief:

GENRE: {genre}
MAIN CHARACTER(S): {characters}
SETTING: {setting}
TONE: {tone}
APPROXIMATE LENGTH: {length_label}
CENTRAL THEME OR MESSAGE: {theme}

Follow all system instructions: begin with a title line, write a complete
story with a real ending, and match the genre, tone, and length as closely
as possible.
""".strip()


print(build_story_prompt("Fantasy", "Mira, a young apprentice mapmaker", "a floating city above the clouds", "Whimsical", "Short (about 300 words)", "courage in the face of the unknown"))


Write an original short story with the following brief:

GENRE: Fantasy
MAIN CHARACTER(S): Mira, a young apprentice mapmaker
SETTING: a floating city above the clouds
TONE: Whimsical
APPROXIMATE LENGTH: Short (about 300 words)
CENTRAL THEME OR MESSAGE: courage in the face of the unknown

Follow all system instructions: begin with a title line, write a complete
story with a real ending, and match the genre, tone, and length as closely
as possible.


## Step 2: A rule-based story engine (default path, no key needed)

Real creative writing from a from-scratch, no-download local model is out of
reach without a real neural network — so instead of pretending to be a tiny
LLM, this default engine is honestly a **template-based generator**: it
picks genre-appropriate opening/conflict/climax/resolution sentences and
plugs in your characters, setting, and theme.

To echo the original app's `temperature` slider, we tie a `temperature`
value (0.2 low → 1.4 high, same range as the original) to how much
**variety** the generator uses: low temperature always picks the same
"safe" first-choice phrasing (deterministic and a bit plain), while high
temperature randomly samples across more phrasing options and adds extra
descriptive "flourish" sentences (more varied and elaborate) — an honest
illustrative analogy to real LLM temperature, not the same mechanism.


In [2]:
import random


def pick(options, rng, temperature, low_threshold=0.35):
    """Low temperature -> always the first ('safe') option. Higher temperature
    -> a pseudo-random pick across all options, echoing more unpredictability."""
    if temperature <= low_threshold:
        return options[0]
    return rng.choice(options)


TITLE_TEMPLATES = [
    "{lead} and the {setting_snip}",
    "The Secret of the {setting_snip}",
    "{lead}'s Journey Through the {setting_snip}",
    "A Tale of the {setting_snip}",
]

OPENERS = [
    "In {setting}, {characters} were about to discover something that would change everything.",
    "{characters} had always known {setting} well, but today felt different.",
    "The story begins in {setting}, where {characters} were going about an ordinary day.",
]

CONFLICTS_BY_GENRE = {
    "Science Fiction": [
        "A strange signal pulsed from the far edge of the system, hinting at danger nobody understood yet.",
        "The systems around {setting} flickered as something unknown approached.",
    ],
    "Fantasy": [
        "An ancient magic stirred in {setting}, restless after centuries of sleep.",
        "A hidden door appeared in {setting} where no door had been before.",
    ],
    "Mystery": [
        "A valuable object had vanished from {setting} without a trace, and the clues made no sense at all.",
        "Someone in {setting} was clearly hiding a secret, and {characters} were determined to find it.",
    ],
    "Adventure": [
        "A weathered map surfaced, pointing {characters} toward a place no one had ever safely returned from.",
        "A sudden storm forced {characters} to take a detour deep into uncharted territory.",
    ],
    "Comedy": [
        "A series of ridiculous misunderstandings began to snowball out of control around {setting}.",
        "A simple plan by {characters} went hilariously wrong within minutes.",
    ],
    "Horror (mild/age-appropriate)": [
        "A chill crept through the air of {setting}, and something unseen seemed to be watching.",
        "Strange noises echoed through {setting} where there should have been only silence.",
    ],
    "Slice of Life": [
        "A small, unexpected moment in {setting} made {characters} stop and think.",
        "An ordinary day in {setting} took an unexpectedly meaningful turn.",
    ],
}

CLIMAX_BY_GENRE = {
    "Science Fiction": [
        "With systems failing all around, {characters} made a bold, split-second decision that changed everything.",
        "In the tense final moments, {characters} found a way to outsmart the unknown threat.",
    ],
    "Fantasy": [
        "Drawing on courage they didn't know they had, {characters} faced the old magic head-on.",
        "At the heart of {setting}, {characters} finally understood what the ancient power truly wanted.",
    ],
    "Mystery": [
        "Piece by piece, {characters} finally assembled the truth, and it was more surprising than anyone expected.",
        "In a tense final confrontation, {characters} revealed exactly what had really happened.",
    ],
    "Adventure": [
        "Against the odds, {characters} pushed through the hardest part of the journey.",
        "With one last effort, {characters} reached the place they'd been searching for all along.",
    ],
    "Comedy": [
        "Somehow, against all logic, {characters} pulled off a plan that really should not have worked.",
        "In a burst of chaos and laughter, everything came together in the most unexpected way.",
    ],
    "Horror (mild/age-appropriate)": [
        "Just as the tension peaked, {characters} discovered the mystery behind the strange happenings in {setting}.",
        "Gathering their courage, {characters} finally confronted the source of the unease in {setting}.",
    ],
    "Slice of Life": [
        "In that quiet moment, {characters} realized something simple but important about their own lives.",
        "It wasn't dramatic, but for {characters}, that afternoon in {setting} changed something inside them.",
    ],
}

FLOURISH_SENTENCES_BY_GENRE = {
    "Science Fiction": [
        "Distant stars glittered through the viewport as {characters} worked against the clock.",
        "The hum of old machinery filled {setting}, steady and strangely comforting.",
    ],
    "Fantasy": [
        "Light shimmered faintly across {setting}, as if the world itself were listening.",
        "Every step {characters} took through {setting} seemed to stir something ancient and half-awake.",
    ],
    "Mystery": [
        "Every detail in {setting} seemed to hold a clue, if only {characters} looked closely enough.",
        "The silence in {setting} felt heavier with every passing minute.",
    ],
    "Adventure": [
        "The path through {setting} twisted on, each turn revealing something new.",
        "{characters} pressed on, undaunted by the challenges {setting} kept throwing their way.",
    ],
    "Comedy": [
        "Naturally, {setting} chose that exact moment to make everything even more complicated.",
        "{characters} exchanged a look that said this was already a story worth telling later.",
    ],
    "Horror (mild/age-appropriate)": [
        "Shadows in {setting} seemed to shift just a little more than they should have.",
        "A distant sound in {setting} made {characters} freeze for a moment, listening.",
    ],
    "Slice of Life": [
        "The ordinary sounds of {setting} carried on around {characters}, softly, like always.",
        "There was something comforting about {setting} on an afternoon like this.",
    ],
}

RESOLUTIONS = [
    "In the end, {characters} realized that {theme} mattered more than anything else.",
    "By the time it was over, {characters} understood {theme} in a way they never had before.",
    "Looking back, {characters} knew this had been a lesson in {theme} they would never forget.",
]

LENGTH_PARAGRAPHS = {
    "Short (about 300 words)": 0,
    "Medium (about 600 words)": 1,
    "Long (about 1000 words)": 3,
}


_DANGLING_TRAILING_WORDS = {
    "the", "a", "an", "of", "in", "on", "above", "through", "with", "and",
    "that", "which", "near", "over", "under", "beside", "by", "for", "to",
}


def _short_phrase(text, fallback, max_words=8, strip_leading_article=False):
    text = (text or "").strip()
    if not text:
        return fallback
    first_clause = text.split(",")[0].split(".")[0]
    words = first_clause.split()
    if strip_leading_article and words and words[0].lower() in ("a", "an", "the"):
        words = words[1:]
    words = words[:max_words]
    # Never end the truncated phrase on a dangling article/preposition/conjunction.
    while words and words[-1].lower() in _DANGLING_TRAILING_WORDS:
        words = words[:-1]
    return " ".join(words) if words else fallback


def _first_name(characters_text, fallback="our hero"):
    text = (characters_text or "").strip()
    if not text:
        return fallback
    first_word = text.split()[0].strip(",.;:")
    if first_word.lower() in ("your", "my"):
        return fallback
    return first_word if first_word else fallback


def generate_rule_based_story(genre, characters, setting, tone, length_label, theme, temperature, rng):
    characters_text = characters.strip() or "a pair of curious friends"
    setting_text = setting.strip() or "a place unlike any other"
    theme_text = theme.strip() or "a personal challenge"

    lead = _first_name(characters_text).title()
    setting_snip = _short_phrase(setting_text, "Unknown", strip_leading_article=True).title()

    conflicts = CONFLICTS_BY_GENRE.get(genre, CONFLICTS_BY_GENRE["Adventure"])
    climaxes = CLIMAX_BY_GENRE.get(genre, CLIMAX_BY_GENRE["Adventure"])
    flourishes = FLOURISH_SENTENCES_BY_GENRE.get(genre, FLOURISH_SENTENCES_BY_GENRE["Adventure"])

    title = pick(TITLE_TEMPLATES, rng, temperature).format(lead=lead, setting_snip=setting_snip)
    opener = pick(OPENERS, rng, temperature).format(setting=setting_text, characters=characters_text)
    conflict = pick(conflicts, rng, temperature).format(setting=setting_text, characters=characters_text)
    climax = pick(climaxes, rng, temperature).format(setting=setting_text, characters=characters_text)
    resolution = pick(RESOLUTIONS, rng, temperature).format(characters=characters_text, theme=theme_text)
    tone_line = f"Throughout, the mood stayed {tone.lower()}, true to the spirit of a {genre.lower()} tale."

    paragraphs = [opener, conflict]

    base_extra = LENGTH_PARAGRAPHS.get(length_label, 1)
    num_extra = min(len(flourishes), base_extra + (1 if temperature > 0.8 else 0))
    if num_extra > 0:
        # Sample WITHOUT replacement so we never repeat the same flourish
        # sentence twice in one story. Low temperature keeps the first
        # ones in list order (deterministic); higher temperature shuffles.
        pool = list(flourishes)
        if temperature > 0.35:
            rng.shuffle(pool)
        for flourish in pool[:num_extra]:
            paragraphs.append(flourish.format(setting=setting_text, characters=characters_text))

    paragraphs.append(climax)
    paragraphs.append(tone_line)
    paragraphs.append(resolution)

    body = "\n\n".join(paragraphs)
    return f"# {title}\n\n{body}"


## Step 3: Optional upgrade — a real OpenAI API key

Leave the field below **blank** to keep using the free rule-based engine
above. If you paste in a real key, Nova will try to write your story using
`gpt-4o-mini` with the exact system prompt and structured brief from Step 1
— and if that call fails for *any* reason, the notebook quietly falls back
to the rule-based engine instead of crashing.


In [3]:
%pip install -q openai
print("openai package ready (only used if you provide an API key below).")


Note: you may need to restart the kernel to use updated packages.
openai package ready (only used if you provide an API key below).


In [4]:
OPENAI_API_KEY = ""  #@param {type:"string"}


In [5]:
def try_openai_story(prompt, temperature, max_tokens):
    """Attempt a real OpenAI call. Returns the story text, or None on any failure."""
    if not OPENAI_API_KEY:
        return None
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content
    except Exception as error:
        print(f"(OpenAI call failed, falling back to the local engine: {error})")
        return None


## Step 4: Putting it together — `generate_story()`

`generate_story()` builds the structured brief, tries the optional OpenAI
path, and falls back to the rule-based engine — this is the single function
you'll call directly, mirroring the original app's "Generate Story" button.


In [6]:
STORY_RNG_SEED = 11
story_rng = random.Random(STORY_RNG_SEED)


def generate_story(genre, characters, setting, tone, length_label, theme, temperature=0.9):
    prompt = build_story_prompt(genre, characters, setting, tone, length_label, theme)
    max_tokens = LENGTHS.get(length_label, 900)

    story = try_openai_story(prompt, temperature, max_tokens)
    if story is None:
        story = generate_rule_based_story(genre, characters, setting, tone, length_label, theme, temperature, story_rng)

    return story


## Step 5: Demo — generate a full story

Concrete example field values, no typing required, so `Runtime → Run all`
always completes with a real generated story.


In [7]:
from IPython.display import Markdown, display

example_genre = "Fantasy"
example_characters = "Mira, a young apprentice mapmaker, and her mechanical owl, Blip"
example_setting = "a floating city that drifts above the clouds"
example_tone = "Whimsical"
example_length = "Medium (about 600 words)"
example_theme = "courage in the face of the unknown"

story_text = generate_story(
    example_genre, example_characters, example_setting,
    example_tone, example_length, example_theme, temperature=0.9,
)

display(Markdown(story_text))


# A Tale of the Floating City That Drifts Above The Clouds

The story begins in a floating city that drifts above the clouds, where Mira, a young apprentice mapmaker, and her mechanical owl, Blip were going about an ordinary day.

A hidden door appeared in a floating city that drifts above the clouds where no door had been before.

Every step Mira, a young apprentice mapmaker, and her mechanical owl, Blip took through a floating city that drifts above the clouds seemed to stir something ancient and half-awake.

Light shimmered faintly across a floating city that drifts above the clouds, as if the world itself were listening.

At the heart of a floating city that drifts above the clouds, Mira, a young apprentice mapmaker, and her mechanical owl, Blip finally understood what the ancient power truly wanted.

Throughout, the mood stayed whimsical, true to the spirit of a fantasy tale.

Looking back, Mira, a young apprentice mapmaker, and her mechanical owl, Blip knew this had been a lesson in courage in the face of the unknown they would never forget.

**Honest note on the rule-based engine:** this default, no-key-required
engine is a **template generator**, not a true language model — it
recombines a fixed bank of genre-flavored sentences with your characters,
setting, and theme rather than freely composing original prose the way
`gpt-4o-mini` would. It reliably produces a complete story with a title,
a beginning, middle, and end every time, but you'll notice the same handful
of sentence patterns if you generate many stories in the same genre. If you
add a real OpenAI key in Step 3, re-run this cell and compare — the story
becomes far more original and varied.


## Step 6: Does `temperature` actually do anything? Compare two versions

Just like the original app's creativity slider, `temperature` here is a
real, observable knob — not just a number for show. Low temperature always
picks the "safe" first-choice phrasing (so it's the same every run); high
temperature samples more randomly and adds extra descriptive sentences.
Generate the *same* brief at both settings and compare.


In [8]:
low_temp_story = generate_story(
    example_genre, example_characters, example_setting,
    example_tone, example_length, example_theme, temperature=0.3,
)
high_temp_story = generate_story(
    example_genre, example_characters, example_setting,
    example_tone, example_length, example_theme, temperature=1.3,
)

print("=" * 70)
print("LOW TEMPERATURE (0.3) — predictable, same phrasing every run")
print("=" * 70)
print(low_temp_story)
print()
print("=" * 70)
print("HIGH TEMPERATURE (1.3) — more varied phrasing and extra descriptive detail")
print("=" * 70)
print(high_temp_story)

assert low_temp_story != high_temp_story, "Expected temperature to visibly change the generated story."
print("\nConfirmed: the two versions differ, just like adjusting temperature should.")


LOW TEMPERATURE (0.3) — predictable, same phrasing every run
# Mira and the Floating City That Drifts Above The Clouds

In a floating city that drifts above the clouds, Mira, a young apprentice mapmaker, and her mechanical owl, Blip were about to discover something that would change everything.

An ancient magic stirred in a floating city that drifts above the clouds, restless after centuries of sleep.

Light shimmered faintly across a floating city that drifts above the clouds, as if the world itself were listening.

Drawing on courage they didn't know they had, Mira, a young apprentice mapmaker, and her mechanical owl, Blip faced the old magic head-on.

Throughout, the mood stayed whimsical, true to the spirit of a fantasy tale.

In the end, Mira, a young apprentice mapmaker, and her mechanical owl, Blip realized that courage in the face of the unknown mattered more than anything else.

HIGH TEMPERATURE (1.3) — more varied phrasing and extra descriptive detail
# The Secret of the Flo

## Step 7: Try it yourself (optional)

Edit the field values below and re-run to generate your own story. Leave
`OPENAI_API_KEY` blank above for the free rule-based engine, or paste in a
real key for a full LLM-written story.


In [9]:
your_story = generate_story(
    genre="Mystery",                              # one of GENRES
    characters="Your character(s) here",
    setting="Your setting here",
    tone="Suspenseful",                            # one of TONES
    length_label="Short (about 300 words)",        # one of LENGTHS keys
    theme="Your theme or message here",
    temperature=0.9,                               # 0.2 (focused) to 1.4 (wild)
)

display(Markdown(your_story))


# Our Hero's Journey Through the Your Setting Here

In Your setting here, Your character(s) here were about to discover something that would change everything.

A valuable object had vanished from Your setting here without a trace, and the clues made no sense at all.

Every detail in Your setting here seemed to hold a clue, if only Your character(s) here looked closely enough.

Piece by piece, Your character(s) here finally assembled the truth, and it was more surprising than anyone expected.

Throughout, the mood stayed suspenseful, true to the spirit of a mystery tale.

Looking back, Your character(s) here knew this had been a lesson in Your theme or message here they would never forget.